<a href="https://colab.research.google.com/github/Nirlss/major/blob/main/absa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import spacy

In [24]:
nlp = spacy.load("en_core_web_sm")

In [25]:
review = nlp("The camera is amazing but the battery does not drains quickly.")

doc = nlp(review)

for token in doc:
    print(token.text,token.pos_)

The DET
camera NOUN
is AUX
amazing ADJ
but CCONJ
the DET
battery NOUN
does AUX
not PART
drains VERB
quickly ADV
. PUNCT


In [26]:
aspects=[]

for token in doc:
  if token.pos_=="NOUN":
    aspects.append(token.text)

print(aspects)

['camera', 'battery']


In [27]:
for chunk in doc.noun_chunks:
  print("Chunk : ",chunk.text)
  print("Root : ",chunk.root.text)
  print("---------------------")

Chunk :  The camera
Root :  camera
---------------------
Chunk :  the battery
Root :  battery
---------------------


In [28]:
def extract_aspects(doc):
    aspects = []

    for chunk in doc.noun_chunks:
        aspect_words = []

        for token in chunk:
            if token.pos_ != "DET":
                aspect_words.append(token.text)

        aspects.append(" ".join(aspect_words))

    return aspects

In [29]:
def extract_opinions(doc):

    opinions = []

    for token in doc:

        if token.pos_ == "ADJ":
            opinions.append(token.text)

        if token.pos_ == "VERB":

            for child in token.children:

                if child.pos_ == "ADV":
                    opinions.append(token.text + " " + child.text)

    return opinions

In [30]:
aspects = extract_aspects(doc)
opinions = extract_opinions(doc)

print("Aspects :", aspects)
print("Opinions:", opinions)

Aspects : ['camera', 'battery']
Opinions: ['amazing', 'drains quickly']


In [31]:
def map_aspect_opinion(doc):
  aspects=[]
  opinions=[]
  mapping={}

  for token in doc:
    if token.pos_=="NOUN":
      aspects.append(token)

    if token.pos_=="ADJ":
      opinions.append(token)

  for aspect in aspects:
    for opinion in opinions:
      if aspect.head==opinion.head:
        mapping[aspect.text]=opinion.text

  return mapping

In [32]:
print(map_aspect_opinion(doc))

{'camera': 'amazing'}


In [37]:
def extract_aspect_opinion_phase4(doc):
  mapping={}

  for token in doc:
    if token.pos_=="NOUN":
      aspect=token.text
      verb=token.head

      opinion_words=[]

      for child in verb.children:
        if child.pos_=="ADJ":
          opinion=child.text

          for grandchild in child.children:
            if grandchild.pos_=="ADV":
              opinion=grandchild.text+" "+grandchild.text

          opinion_words.append(opinion)

        elif child.pos_=="ADV":
          negation=False
          for grandchild in verb.children:
            if grandchild.text=="not":
              negation=True
              opinion = verb.text+" "+child.text


          if negation:
            opinion="not "+opinion

          opinion_words.append(opinion)

      mapping[aspect]=" ".join(opinion_words)

  return mapping


In [38]:


print(extract_aspect_opinion_phase4(doc))

{'camera': 'amazing', 'battery': 'not drains quickly'}


In [44]:
positive_words = [
    "good",
    "great",
    "excellent",
    "amazing",
    "beautiful",
    "perfect",
    "fast",
    "smoothly"
]

negative_words = [
    "bad",
    "poor",
    "terrible",
    "worst",
    "slow",
    "drain",
    "drains",
    "expensive"
]

In [45]:
def get_sentiment(opinion):
  words = opinion.split()
  sentiment="Neutral"
  for word in words:
    if word in positive_words:
      sentiment="positive"
    elif word in negative_words:
      sentiment="negative"
  if "not" in words:
    if sentiment=="positive":
      sentiment="negative"
    elif sentiment=="negative":
      sentiment="positive"
  return sentiment

In [47]:
print(get_sentiment("not bad"))

positive
